# Bidirectional MedSAM2 — Colab Notebook

| Section | Content |
|---|---|
| 1 – Setup | Install MedSAM2, check device, download checkpoint |
| 2 – Load data | Load a sample CT lesion NPZ volume |
| 3 – Causal inference | Standard SAM2 forward sweep (`propagate_in_video`) |
| 4 – Bidirectional inference | Two-pass bidirectional sweep (`propagate_in_video_bidirectional`) |
| 5 – Visualise & compare | Overlay masks, compare consistency between passes |
| 6 – Fine-tune | Lightweight fine-tune that adds the slice consistency loss |
| 7 – Post-finetune check | Sanity inference with fine-tuned weights |

**Runtime recommendation:** GPU (T4 or better).  
The notebook also runs on CPU / Apple Silicon MPS — it will just be slower.

## Section 1 — Environment Setup

Run this cell once after connecting to a Colab runtime.  
Re-run with *Runtime → Restart and run all* if you change the checkpoint path.

In [ ]:
import subprocess, sys, os

def _colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _colab():
    # Clone repo and install (skip if already cloned)
    if not os.path.isdir('MedSAM2'):
        subprocess.run(
            ['git', 'clone', 'https://github.com/bowang-lab/MedSAM2.git'],
            check=True,
        )
    os.chdir('MedSAM2')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', '.[dev]',
         '--quiet', '--no-build-isolation'],
        env={**os.environ, 'SAM2_BUILD_CUDA': '0'},
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'],
        check=True,
    )
    print('MedSAM2 installed.')
else:
    print('Not on Colab - assuming MedSAM2 is already installed in the current env.')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'MPS  available : {torch.backends.mps.is_available()}')
import sam2  # noqa: F401
print('sam2 import OK')

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from sam2.build_sam import get_best_available_device
from sam2.bidirectional_video_predictor import (
    build_bidir_sam2_video_predictor_npz,
    slice_consistency_loss,
)

# Device
DEVICE = get_best_available_device()
AUTOCAST_DEVICE = 'cuda' if DEVICE == 'cuda' else 'cpu'
print(f'Using device: {DEVICE}  (autocast: {AUTOCAST_DEVICE})')

torch.set_float32_matmul_precision('high')
torch.manual_seed(2024)
if torch.cuda.is_available():
    torch.cuda.manual_seed(2024)
np.random.seed(2024)

In [ ]:
from huggingface_hub import hf_hub_download

os.makedirs('./checkpoints', exist_ok=True)

CKPT_PATH = hf_hub_download(
    repo_id='wanglab/MedSAM2',
    filename='medsam2_FLARE25_RECIST_baseline.pt',
    cache_dir='./checkpoints',
)

script_dir = os.path.abspath('.')
MODEL_CFG_ABS = '//' + os.path.join(script_dir, 'sam2', 'configs', 'sam2.1_hiera_t512.yaml')

print(f'Checkpoint : {CKPT_PATH}')
print(f'Config     : {MODEL_CFG_ABS}')

## Section 2 — Load Data

Each NPZ contains:
- `imgs` — 3-D CT volume `[D, H, W]`, uint8, range `[0, 255]`
- `recist` — RECIST annotation (line drawn on key slice)
- `spacing` — voxel spacing in mm

Point `NPZ_PATH` at your own file to use custom data.

In [ ]:
from glob import glob

NPZ_DIR = './data/validation_public_npz'
npz_files = sorted(glob(os.path.join(NPZ_DIR, '*.npz')))
if not npz_files:
    NPZ_DIR = './data/RECIST_train_npz'
    npz_files = sorted(glob(os.path.join(NPZ_DIR, '*.npz')))

NPZ_PATH = npz_files[0]
print(f'Using: {NPZ_PATH}')

npz_data = np.load(NPZ_PATH, allow_pickle=True)
print(f'Keys: {list(npz_data.files)}')

img_3D_ori = npz_data['imgs']       # [D, H, W]  uint8
recist     = npz_data['recist']
spacing    = npz_data['spacing']

D, H, W = img_3D_ori.shape
print(f'Volume shape : {img_3D_ori.shape}  dtype={img_3D_ori.dtype}')
print(f'Spacing (mm) : {spacing}')
print(f'RECIST unique: {np.unique(recist)}')

In [ ]:
def resize_grayscale_to_rgb_and_resize(array, image_size=512):
    """[D, H, W] uint8  ->  [D, 3, image_size, image_size] float32."""
    d = array.shape[0]
    out = np.zeros((d, 3, image_size, image_size), dtype=np.float32)
    for i in range(d):
        img = Image.fromarray(array[i].astype(np.uint8)).convert('RGB')
        img = img.resize((image_size, image_size))
        out[i] = np.array(img).transpose(2, 0, 1)
    return out


def preprocess_volume(img_3d, device, image_size=512):
    """Normalise and move to device.  Returns [D, 3, H, W] tensor."""
    assert img_3d.max() < 256
    x = resize_grayscale_to_rgb_and_resize(img_3d, image_size)
    x = torch.from_numpy(x / 255.0).to(device)
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].to(device)
    std  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].to(device)
    return (x - mean) / std


def get_recist_prompt(recist_vol, label=1, n_points=5):
    """Return (z_mid, points [N,2], labels [N]) from RECIST annotation."""
    z_indices = np.unique(np.where(recist_vol == label)[0])
    assert len(z_indices) == 1, f'Expected 1 RECIST slice, got {z_indices}'
    z_mid = int(z_indices[0])
    ys, xs = np.where(recist_vol[z_mid] > 0)
    coords = np.stack([xs, ys], axis=1)
    n = min(n_points, len(coords))
    idx = np.linspace(0, len(coords) - 1, n, dtype=int)
    return z_mid, coords[idx].astype(np.float32), np.ones(n, dtype=np.int32)


def show_mask(mask, ax, color=None, alpha=0.45):
    c = color if color is not None else np.array([1.0, 0.95, 0.1])
    rgba = np.concatenate([c, [alpha]])
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * rgba.reshape(1, 1, 4))


def show_points(points, ax, marker_size=60):
    ax.scatter(points[:, 0], points[:, 1], s=marker_size,
               c='red', marker='*', edgecolors='white', linewidths=0.8)


def seg_to_logits(seg):
    """Binary uint8 mask volume -> approximate logit tensor [D,1,H,W]."""
    t = torch.from_numpy(seg.astype(np.float32)).unsqueeze(1)
    return t * 10.0 - 5.0


print('Helpers defined.')

## Section 3 — Build Predictor & Causal Inference

`BidirectionalSAM2VideoPredictorNPZ` is a drop-in replacement for the original
predictor. Here we use it in **causal mode** (standard forward + reverse sweep)
as the baseline.

In [ ]:
predictor = build_bidir_sam2_video_predictor_npz(
    config_file=MODEL_CFG_ABS,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
)
print(f'Predictor type : {type(predictor).__name__}')
print(f'num_maskmem    : {predictor.num_maskmem}')

In [ ]:
import time

img_tensor = preprocess_volume(img_3D_ori, DEVICE)   # [D, 3, 512, 512]
video_h, video_w = img_3D_ori.shape[1], img_3D_ori.shape[2]

unique_labs = np.unique(recist)
unique_labs = unique_labs[unique_labs != 0]
lab = unique_labs[0]
z_mid, points, point_labels = get_recist_prompt(recist, label=int(lab))
print(f'Key slice: {z_mid},  #points: {len(points)}')

segs_causal = np.zeros(img_3D_ori.shape, dtype=np.uint8)

t0 = time.time()
with torch.inference_mode(), torch.autocast(AUTOCAST_DEVICE, dtype=torch.bfloat16):
    inf_state = predictor.init_state(img_tensor, video_h, video_w)
    _, _, logits = predictor.add_new_points_or_box(
        inference_state=inf_state, frame_idx=z_mid, obj_id=1,
        points=points, labels=point_labels,
    )
    mask_prompt = (logits[0] > 0.0).squeeze(0).cpu().numpy().astype(np.uint8)
    predictor.add_new_mask(inf_state, frame_idx=z_mid, obj_id=1, mask=mask_prompt)
    segs_causal[z_mid] = mask_prompt

    for fi, _, mlogits in predictor.propagate_in_video(inf_state, start_frame_idx=z_mid, reverse=False):
        segs_causal[fi] = (mlogits[0] > 0.0).cpu().numpy()[0].astype(np.uint8)
    predictor.reset_state(inf_state)

    inf_state = predictor.init_state(img_tensor, video_h, video_w)
    predictor.add_new_mask(inf_state, frame_idx=z_mid, obj_id=1, mask=mask_prompt)
    for fi, _, mlogits in predictor.propagate_in_video(inf_state, start_frame_idx=z_mid, reverse=True):
        segs_causal[fi] = (mlogits[0] > 0.0).cpu().numpy()[0].astype(np.uint8)
    predictor.reset_state(inf_state)

t_causal = time.time() - t0
print(f'Causal inference done in {t_causal:.2f} s')
print(f'Causal mask voxels: {segs_causal.sum()}')

## Section 4 — Bidirectional Inference

`propagate_in_video_bidirectional` runs a two-pass algorithm:

1. **Pass 1** — Standard causal forward sweep producing initial masks for all slices.
2. **Pass 2** — Each non-conditioning slice is re-run with memories from *both*
   past **and** future slices from Pass 1.

The generator yields `(frame_idx, obj_ids, masks)` — same API as `propagate_in_video`.

In [ ]:
segs_bidir = np.zeros(img_3D_ori.shape, dtype=np.uint8)

t0 = time.time()
with torch.inference_mode(), torch.autocast(AUTOCAST_DEVICE, dtype=torch.bfloat16):
    inf_state = predictor.init_state(img_tensor, video_h, video_w)
    _, _, logits = predictor.add_new_points_or_box(
        inference_state=inf_state, frame_idx=z_mid, obj_id=1,
        points=points, labels=point_labels,
    )
    mask_prompt = (logits[0] > 0.0).squeeze(0).cpu().numpy().astype(np.uint8)
    predictor.add_new_mask(inf_state, frame_idx=z_mid, obj_id=1, mask=mask_prompt)

    for fi, _, mlogits in predictor.propagate_in_video_bidirectional(inf_state):
        segs_bidir[fi] = (mlogits[0] > 0.0).cpu().numpy()[0].astype(np.uint8)

    predictor.reset_state(inf_state)

t_bidir = time.time() - t0
print(f'Bidirectional done in {t_bidir:.2f} s  ({t_bidir/t_causal:.1f}x causal time)')
print(f'Bidirectional mask voxels: {segs_bidir.sum()}')

## Section 5 — Visualise & Compare

- **Left** (blue): causal forward + reverse
- **Right** (yellow): bidirectional two-pass

We also compare slice consistency loss — lower means smoother segmentation across slices.

In [ ]:
BLUE   = np.array([0.2, 0.4, 1.0])
YELLOW = np.array([1.0, 0.95, 0.1])

n_show  = min(5, D)
indices = np.linspace(max(0, z_mid - 2), min(D - 1, z_mid + 2), n_show, dtype=int)

fig, axes = plt.subplots(n_show, 2, figsize=(10, 3 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for col, (segs, title, color) in enumerate([
    (segs_causal, 'Causal (fwd+rev)',       BLUE),
    (segs_bidir,  'Bidirectional (2-pass)', YELLOW),
]):
    for row, zi in enumerate(indices):
        ax = axes[row, col]
        ax.imshow(img_3D_ori[zi], cmap='gray', vmin=0, vmax=255)
        if segs[zi].any():
            show_mask(segs[zi][np.newaxis], ax, color=color)
        if int(zi) == z_mid:
            show_points(points, ax)
            ax.set_title(f'{title}  [z={zi} <- key]', fontsize=9)
        else:
            ax.set_title(f'{title}  [z={zi}]', fontsize=9)
        ax.axis('off')

plt.suptitle('Causal vs Bidirectional', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
cons_causal = slice_consistency_loss(seg_to_logits(segs_causal)).item()
cons_bidir  = slice_consistency_loss(seg_to_logits(segs_bidir)).item()

print('Slice consistency loss (lower = smoother):')
print(f'  Causal        : {cons_causal:.4f}')
print(f'  Bidirectional : {cons_bidir:.4f}')
pct = (cons_causal - cons_bidir) / (cons_causal + 1e-8) * 100
print(f'  Change        : {pct:+.1f}%')

def dice_coeff(a, b):
    inter = (a & b).sum()
    return 2 * inter / (a.sum() + b.sum() + 1e-8)

d = dice_coeff(segs_causal > 0, segs_bidir > 0)
print(f'\nDice (causal vs bidir): {d:.3f}  (1.0=identical, lower=more refinement)')

## Section 6 — Fine-tuning with Slice Consistency Loss

Total loss:
$$\mathcal{L} = \mathcal{L}_{\text{mask}} + \mathcal{L}_{\text{iou}} + \lambda \cdot \mathcal{L}_{\text{cons}}$$

**Frozen:** image encoder (`model.image_encoder`)  
**Trained:** mask decoder + memory encoder + memory attention

> Tip: On Colab T4 (16 GB VRAM) `batch_size=1` (one 3-D volume per step) fits comfortably.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Hyper-parameters
LR           = 1e-5
CONS_WEIGHT  = 0.1     # lambda for consistency term
NUM_EPOCHS   = 3       # increase for real fine-tuning
IMAGE_SIZE   = 512
GRAD_CLIP    = 1.0

# Freeze image encoder
for name, param in predictor.named_parameters():
    param.requires_grad = 'image_encoder' not in name

trainable = sum(p.numel() for p in predictor.parameters() if p.requires_grad)
total     = sum(p.numel() for p in predictor.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, predictor.parameters()),
    lr=LR, weight_decay=1e-4,
)

In [ ]:
class MedSAM2NPZDataset(Dataset):
    """Loads NPZ volumes. Each item is one 3-D CT volume."""

    def __init__(self, npz_dir, image_size=512):
        self.files = sorted(glob(os.path.join(npz_dir, '*.npz')))
        self.image_size = image_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data    = np.load(self.files[idx], allow_pickle=True)
        img_3d  = data['imgs']
        recist  = data['recist']
        H, W    = img_3d.shape[1], img_3d.shape[2]

        imgs_norm = preprocess_volume(img_3d, device='cpu', image_size=self.image_size)

        labs = np.unique(recist)
        labs = labs[labs != 0]
        lab  = int(labs[0]) if len(labs) > 0 else 1

        z_mid, pts, lbls = get_recist_prompt(recist, label=lab)

        gt = np.zeros(img_3d.shape, dtype=np.uint8)
        gt[z_mid] = (recist[z_mid] == lab).astype(np.uint8)

        return {
            'imgs':     imgs_norm,
            'gt_masks': torch.from_numpy(gt).unsqueeze(1).float(),
            'z_mid':    z_mid,
            'points':   torch.from_numpy(pts),
            'labels':   torch.from_numpy(lbls),
            'video_hw': (H, W),
        }


train_dataset = MedSAM2NPZDataset('./data/RECIST_train_npz',      image_size=IMAGE_SIZE)
val_dataset   = MedSAM2NPZDataset('./data/validation_public_npz', image_size=IMAGE_SIZE)
train_loader  = DataLoader(train_dataset, batch_size=1, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=1, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}')

In [ ]:
def soft_dice_loss(logits, targets):
    pred  = torch.sigmoid(logits).flatten(1)
    tgt   = targets.flatten(1)
    inter = (pred * tgt).sum(1)
    denom = pred.sum(1) + tgt.sum(1)
    return (1.0 - (2.0 * inter + 1.0) / (denom + 1.0)).mean()


def run_one_volume(batch, predictor, device, autocast_device, cons_weight=0.1):
    imgs     = batch['imgs'][0].to(device)
    gt_masks = batch['gt_masks'][0].to(device)
    z_mid    = int(batch['z_mid'][0])
    pts      = batch['points'][0].cpu().numpy()
    lbls     = batch['labels'][0].cpu().numpy()
    H, W     = int(batch['video_hw'][0][0]), int(batch['video_hw'][0][1])

    with torch.autocast(autocast_device, dtype=torch.bfloat16):
        inf_state = predictor.init_state(imgs, H, W)
        _, _, logits_init = predictor.add_new_points_or_box(
            inference_state=inf_state, frame_idx=z_mid, obj_id=1,
            points=pts, labels=lbls,
        )
        mp = (logits_init[0] > 0.0).squeeze(0).detach().cpu().numpy().astype(np.uint8)
        predictor.add_new_mask(inf_state, frame_idx=z_mid, obj_id=1, mask=mp)

        all_logits = {}
        for fi, _, mlogits in predictor.propagate_in_video_bidirectional(inf_state):
            all_logits[fi] = mlogits[0]
        predictor.reset_state(inf_state)

    logits_stack = torch.cat([all_logits[fi] for fi in sorted(all_logits)], dim=0)

    H_out, W_out = logits_stack.shape[-2:]
    gt_r = F.interpolate(gt_masks, size=(H_out, W_out), mode='nearest') \
           if (H_out, W_out) != (gt_masks.shape[-2], gt_masks.shape[-1]) else gt_masks

    l_mask = soft_dice_loss(logits_stack, gt_r)
    l_iou  = F.binary_cross_entropy_with_logits(logits_stack.float(), gt_r.float())
    l_cons = slice_consistency_loss(logits_stack.float(), weight=cons_weight)
    total  = l_mask + l_iou + l_cons

    return {'total': total, 'mask': l_mask.item(), 'iou': l_iou.item(), 'cons': l_cons.item()}


print('Loss helpers defined.')

In [ ]:
history = {'train_total': [], 'train_cons': [], 'val_total': [], 'val_cons': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    predictor.train()
    tm = {'total': 0., 'mask': 0., 'iou': 0., 'cons': 0.}

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [train]', leave=False):
        optimizer.zero_grad()
        losses = run_one_volume(batch, predictor, DEVICE, AUTOCAST_DEVICE, CONS_WEIGHT)
        losses['total'].backward()
        nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, predictor.parameters()), GRAD_CLIP
        )
        optimizer.step()
        for k in tm:
            tm[k] += losses[k] if isinstance(losses[k], float) else losses[k].item()

    n = len(train_loader)
    for k in tm:
        tm[k] /= n
    history['train_total'].append(tm['total'])
    history['train_cons'].append(tm['cons'])

    # Validate
    predictor.eval()
    vm = {'total': 0., 'cons': 0.}

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [val]', leave=False):
            losses = run_one_volume(batch, predictor, DEVICE, AUTOCAST_DEVICE, CONS_WEIGHT)
            vm['total'] += losses['total'].item() if hasattr(losses['total'], 'item') else losses['total']
            vm['cons']  += losses['cons']

    nv = max(len(val_loader), 1)
    for k in vm:
        vm[k] /= nv
    history['val_total'].append(vm['total'])
    history['val_cons'].append(vm['cons'])

    print(f'Epoch {epoch:>2}  '
          f'train total={tm["total"]:.4f}  mask={tm["mask"]:.4f}  cons={tm["cons"]:.4f}  ||'
          f'  val total={vm["total"]:.4f}  cons={vm["cons"]:.4f}')

print('Fine-tuning complete.')

In [ ]:
epochs = list(range(1, NUM_EPOCHS + 1))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_total'], 'b-o', label='train total')
axes[0].plot(epochs, history['val_total'],   'r-o', label='val total')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Total loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['train_cons'], 'b-o', label='train cons')
axes[1].plot(epochs, history['val_cons'],   'r-o', label='val cons')
axes[1].set(xlabel='Epoch', ylabel='Consistency loss', title='Slice consistency loss')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('./checkpoints/finetuned', exist_ok=True)
save_path = './checkpoints/finetuned/medsam2_bidir_finetuned.pt'

torch.save(
    {
        'model_state_dict': predictor.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': NUM_EPOCHS,
        'history': history,
        'config': {
            'model_cfg': MODEL_CFG_ABS,
            'lr': LR,
            'cons_weight': CONS_WEIGHT,
        },
    },
    save_path,
)
print(f'Checkpoint saved: {save_path}')
print()
print('# How to reload:')
print('# predictor = build_bidir_sam2_video_predictor_npz(MODEL_CFG_ABS, device=DEVICE)')
print('# ckpt = torch.load(save_path, map_location=DEVICE, weights_only=True)')
print('# predictor.load_state_dict(ckpt["model_state_dict"])')
print('# predictor.eval()')

## Section 7 — Post-fine-tune Inference (sanity check)

Re-run bidirectional inference with the fine-tuned weights and compare the
slice consistency score against the pre-fine-tuned result.

In [ ]:
segs_finetuned = np.zeros(img_3D_ori.shape, dtype=np.uint8)

predictor.eval()
with torch.inference_mode(), torch.autocast(AUTOCAST_DEVICE, dtype=torch.bfloat16):
    inf_state = predictor.init_state(img_tensor, video_h, video_w)
    _, _, logits = predictor.add_new_points_or_box(
        inference_state=inf_state, frame_idx=z_mid, obj_id=1,
        points=points, labels=point_labels,
    )
    mp = (logits[0] > 0.0).squeeze(0).cpu().numpy().astype(np.uint8)
    predictor.add_new_mask(inf_state, frame_idx=z_mid, obj_id=1, mask=mp)

    for fi, _, mlogits in predictor.propagate_in_video_bidirectional(inf_state):
        segs_finetuned[fi] = (mlogits[0] > 0.0).cpu().numpy()[0].astype(np.uint8)

    predictor.reset_state(inf_state)

cons_ft = slice_consistency_loss(seg_to_logits(segs_finetuned)).item()
print(f'Consistency -- bidir pretrained : {cons_bidir:.4f}')
print(f'Consistency -- bidir fine-tuned  : {cons_ft:.4f}')

GREEN = np.array([0.1, 0.9, 0.2])
fig, axes = plt.subplots(n_show, 2, figsize=(10, 3 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]
for col, (segs, title, color) in enumerate([
    (segs_bidir,     'Bidir - pretrained', YELLOW),
    (segs_finetuned, 'Bidir - fine-tuned', GREEN),
]):
    for row, zi in enumerate(indices):
        ax = axes[row, col]
        ax.imshow(img_3D_ori[zi], cmap='gray', vmin=0, vmax=255)
        if segs[zi].any():
            show_mask(segs[zi][np.newaxis], ax, color=color)
        if int(zi) == z_mid:
            show_points(points, ax)
        ax.set_title(f'{title}  [z={zi}]', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.show()